# Previsão de Direção do Preço (PETR4) com Machine Learning

**Pergunta de negócio:** o fechamento da PETR4 em 5 dias úteis será maior do que o
fechamento de hoje? (classificação binária: Sobe / Cai)

**Estrutura:**
1. Configuração e constantes
2. Coleta de dados (preço, petróleo, câmbio)
3. Engenharia de features
4. Definição do target
5. Análise exploratória
6. Baseline
7. Modelos e validação (TimeSeriesSplit)
8. Otimização de hiperparâmetros
9. Importância das features
10. Backtest da estratégia
11. Conclusões


## 1. Configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score
from xgboost import XGBClassifier

RANDOM_STATE = 7
TICKER = "PETR4.SA"
PERIOD = "10y"
HORIZON_DIAS = 5
TEST_SIZE = 0.2

np.random.seed(RANDOM_STATE)

## 2. Coleta de dados

Além do preço do ativo, trazemos duas variáveis de contexto macro relevantes para
uma petroleira: o preço do petróleo Brent e o câmbio USD/BRL. Cada série é
deslocada em um dia (`shift(1)`) porque, no momento em que a previsão seria feita,
o fechamento de "hoje" dessas séries ainda não está disponível.

In [ ]:
def carregar_precos(ticker: str, period: str) -> pd.DataFrame:
    """Baixa OHLCV de um ticker via yfinance e normaliza as colunas."""
    df = yf.download(ticker, period=period, progress=False)
    if df.empty:
        raise ValueError(f"Nenhum dado retornado para o ticker '{ticker}'.")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
    return df


def carregar_serie_retorno(ticker: str, period: str) -> pd.Series:
    """Baixa o fechamento de um ticker e retorna a série de retorno percentual diário."""
    serie = yf.download(ticker, period=period, progress=False)["Close"]
    return serie.pct_change()

In [ ]:
precos = carregar_precos(TICKER, PERIOD)

contexto = pd.DataFrame(index=precos.index)
contexto["ibov_return"] = carregar_serie_retorno("^BVSP", PERIOD).shift(1)
contexto["brent_return"] = carregar_serie_retorno("BZ=F", PERIOD).shift(1)
contexto["usdbrl_return"] = carregar_serie_retorno("BRL=X", PERIOD).shift(1)

df = precos.join(contexto)
df.head()

## 3. Engenharia de features

Cada função calcula um grupo de features e retorna apenas as colunas novas, o que
facilita testar cada bloco isoladamente e documentar a intenção de cada um.

Todas as janelas (`rolling`, `shift`, `ewm`) olham apenas para o passado em relação
a cada linha, então não há vazamento de dados nesta etapa.

In [ ]:
def features_tendencia(close: pd.Series) -> pd.DataFrame:
    """Médias móveis de curto/longo prazo e a razão entre elas."""
    ma5 = close.rolling(5).mean()
    ma20 = close.rolling(20).mean()
    ma50 = close.rolling(50).mean()
    ma200 = close.rolling(200).mean()
    return pd.DataFrame({
        "ma_ratio": ma5 / ma20,
        "trend": ma50 / ma200,          # acima de 1 = tá em alta
        "dist_ma20": (close - ma20) / ma20,
    })


def features_volatilidade(close: pd.Series, high: pd.Series, low: pd.Series,
                           retorno: pd.Series) -> pd.DataFrame:
    """Volatilidade histórica, volatilidade relativa e ATR normalizado."""
    volatility = retorno.rolling(10).std()
    vol_ratio = volatility / retorno.rolling(60).std()

    high_low = high - low
    high_close = (high - close.shift()).abs()
    low_close = (low - close.shift()).abs()
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr_pct = true_range.rolling(14).mean() / close

    return pd.DataFrame({
        "volatility": volatility,
        "vol_ratio": vol_ratio,
        "atr_pct": atr_pct,
    })


def features_momentum(close: pd.Series) -> pd.DataFrame:
    """RSI, MACD (histograma) e momentum de 5 dias."""
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rsi = 100 - (100 / (1 + gain / loss))

    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    macd_hist = macd_line - signal_line

    momentum = close / close.shift(5) - 1

    return pd.DataFrame({
        "rsi": rsi,
        "macd_hist": macd_hist,
        "momentum": momentum,
    })


def features_volume_e_candle(open_: pd.Series, high: pd.Series, low: pd.Series,
                              close: pd.Series, volume: pd.Series) -> pd.DataFrame:
    """Volume relativo e formato do candle (corpo e sombra)."""
    volume_ma = volume.rolling(5).mean()
    return pd.DataFrame({
        "volume_ratio": volume / volume_ma,
        "candle_body": (close - open_) / open_,
        "candle_shadow": (high - low) / open_,
    })


def features_bollinger(close: pd.Series, janela: int = 20) -> pd.DataFrame:
    """Posição do preço relativa às Bandas de Bollinger (-1 a 1, aproximadamente)."""
    media = close.rolling(janela).mean()
    desvio = close.rolling(janela).std()
    return pd.DataFrame({
        "boll_position": (close - media) / (2 * desvio),
    })

In [ ]:
df["return"] = df["Close"].pct_change()

df = df.join(features_tendencia(df["Close"]))
df = df.join(features_volatilidade(df["Close"], df["High"], df["Low"], df["return"]))
df = df.join(features_momentum(df["Close"]))
df = df.join(features_volume_e_candle(df["Open"], df["High"], df["Low"], df["Close"], df["Volume"]))
df = df.join(features_bollinger(df["Close"]))
df["day_of_week"] = df.index.dayofweek

FEATURES = [
    "return", "ma_ratio", "volatility", "vol_ratio", "dist_ma20", "trend",
    "rsi", "momentum", "volume_ratio", "candle_body", "candle_shadow",
    "ibov_return", "brent_return", "usdbrl_return",
    "macd_hist", "boll_position", "atr_pct", "day_of_week",
]

df[FEATURES].tail()

## 4. Definição do target

In [ ]:
def construir_target(close: pd.Series, horizonte: int) -> pd.Series:
    """1 se o fechamento em `horizonte` dias úteis for maior que o de hoje, senão 0."""
    return (close.shift(-horizonte) > close).astype(int)

In [ ]:
df["target"] = construir_target(df["Close"], HORIZON_DIAS)
df.dropna(inplace=True)

print(f"Linhas após remover NaNs: {len(df)}")
print(f"Distribuição do target:\n{df['target'].value_counts(normalize=True)}")

In [ ]:
X = df[FEATURES]
y = df["target"]

split = int(len(df) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras")

## 5. Análise exploratória

Correlação entre as features: valores muito próximos de 1 ou -1 indicam informação
redundante, o que pode ser simplificado depois se a importância das features (seção
9) confirmar baixo poder preditivo individual.

In [ ]:
correlacao = X.corr()

plt.figure(figsize=(11, 9))
plt.imshow(correlacao, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlação")
plt.xticks(range(len(FEATURES)), FEATURES, rotation=90)
plt.yticks(range(len(FEATURES)), FEATURES)
plt.title("Matriz de correlação entre features")
plt.tight_layout()
plt.show()

### Features realmente separam as classes?

Correlação entre features não diz se elas ajudam a distinguir Sobe de Cai. Boxplots
comparando a distribuição de cada feature por classe do target respondem isso de
forma mais direta: se as caixas de "Sobe" e "Cai" se sobrepõem quase totalmente,
essa feature sozinha carrega pouco sinal.

In [ ]:
features_para_inspecionar = ["rsi", "momentum", "macd_hist", "boll_position", "brent_return", "trend"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, features_para_inspecionar):
    df.boxplot(column=feat, by="target", ax=ax)
    ax.set_title(feat)
    ax.set_xlabel("target (0 = Cai, 1 = Sobe)")
plt.suptitle("Distribuição das features por classe do target")
plt.tight_layout()
plt.show()

## 6. Baseline

Referência mínima de comparação: prever sempre a classe majoritária. Todo modelo
precisa superar isso de forma consistente, no holdout e no CV, para ser
considerado útil.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
acc_baseline = accuracy_score(y_test, baseline.predict(X_test))
print(f"Baseline (classe majoritária): {acc_baseline:.2%}")

## 7. Modelos e validação

In [ ]:
def avaliar_modelo(nome: str, model, X_train, y_train, X_test, y_test) -> dict:
    """Treina, prediz no holdout e retorna métricas + o classification report impresso."""
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metricas = {
        "accuracy": accuracy_score(y_test, pred),
        "precision_sobe": precision_score(y_test, pred, pos_label=1, zero_division=0),
    }
    print(f"--- {nome} ---")
    print(f"Accuracy: {metricas['accuracy']:.2%}")
    print(classification_report(y_test, pred, target_names=["Cai (0)", "Sobe (1)"], zero_division=0))
    return metricas


def validar_series_temporais(nome: str, model_factory, X: pd.DataFrame, y: pd.Series,
                              n_splits: int = 5) -> list[float]:
    """Avalia um modelo com TimeSeriesSplit, treinando um modelo novo em cada fold."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for train_idx, test_idx in tscv.split(X):
        modelo = model_factory()
        modelo.fit(X.iloc[train_idx], y.iloc[train_idx])
        scores.append(accuracy_score(y.iloc[test_idx], modelo.predict(X.iloc[test_idx])))
    print(f"{nome}: CV médio = {np.mean(scores):.2%} +/- {np.std(scores):.2%}")
    return scores

In [ ]:
MODELOS = {
    "Decision Tree": lambda: DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=3, random_state=RANDOM_STATE
    ),
    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=200, max_depth=5, min_samples_leaf=15, random_state=RANDOM_STATE
    ),
    "XGBoost": lambda: XGBClassifier(
        n_estimators=500, max_depth=2, learning_rate=0.02,
        subsample=0.6, colsample_bytree=0.6, min_child_weight=15,
        gamma=0.5, reg_alpha=0.5, reg_lambda=2.0, random_state=RANDOM_STATE,
    ),
}

resultados_holdout = {}
for nome, factory in MODELOS.items():
    resultados_holdout[nome] = avaliar_modelo(nome, factory(), X_train, y_train, X_test, y_test)

In [ ]:
resultados_cv = {
    nome: validar_series_temporais(nome, factory, X, y)
    for nome, factory in MODELOS.items()
}

## 8. Otimização de hiperparâmetros

`GridSearchCV` com `TimeSeriesSplit` como estratégia de validação: a busca nunca usa
dados futuros para escolher parâmetros avaliados em dados passados. Aplicado ao
Random Forest como exemplo, o mesmo padrão se aplica aos demais modelos.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

grade_parametros = {
    "n_estimators": [100, 200, 400],
    "max_depth": [3, 5, 7],
    "min_samples_leaf": [5, 15, 30],
}

busca = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    grade_parametros,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1,
)
busca.fit(X_train, y_train)

modelo_otimizado = busca.best_estimator_
acc_otimizado = accuracy_score(y_test, modelo_otimizado.predict(X_test))

print(f"Melhores parâmetros: {busca.best_params_}")
print(f"Melhor accuracy média (CV): {busca.best_score_:.2%}")
print(f"Accuracy no holdout com parâmetros otimizados: {acc_otimizado:.2%}")

## 8.1 Por que o Decision Tree ficou abaixo do baseline?

O Decision Tree teve o pior resultado em CV (49.24%) e o maior desvio padrão entre
folds (3.73%) dos três modelos: ambos são sintomas clássicos de uma árvore única
com overfitting: ela memoriza padrões específicos do conjunto de treino que não se
repetem no teste, em vez de aprender uma regra que generaliza.

Três checagens para confirmar isso:

In [ ]:
# check 1: será que ele só decorou o treino? se a accuracy do treino for
# bem maior que a do teste, é overfitting na cara.
arvore_diagnostico = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=RANDOM_STATE)
arvore_diagnostico.fit(X_train, y_train)

acc_treino = accuracy_score(y_train, arvore_diagnostico.predict(X_train))
acc_teste = accuracy_score(y_test, arvore_diagnostico.predict(X_test))

print(f"Accuracy no treino: {acc_treino:.2%}")
print(f"Accuracy no teste:  {acc_teste:.2%}")
print(f"Gap (treino - teste): {acc_treino - acc_teste:.2%}")

In [ ]:
# check 2: quantas folhas essa árvore tá criando? com min_samples_leaf=3
# ela pode acabar fazendo regras específicas demais, que não generalizam.
print(f"Número de folhas: {arvore_diagnostico.get_n_leaves()}")
print(f"Profundidade real atingida: {arvore_diagnostico.get_depth()}")
print(f"Amostras de treino por folha (média): {len(X_train) / arvore_diagnostico.get_n_leaves():.1f}")

In [ ]:
# check 3: essa árvore é instável? treino em pedaços diferentes do passado
# e testo sempre no mesmo período. se a accuracy pular muito de um treino
# pro outro, é sinal de variância alta.
janelas_treino = [0.5, 0.65, 0.8]
for fracao in janelas_treino:
    corte = int(len(X_train) * fracao)
    arvore_tmp = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=RANDOM_STATE)
    arvore_tmp.fit(X_train.iloc[-corte:], y_train.iloc[-corte:])
    acc_tmp = accuracy_score(y_test, arvore_tmp.predict(X_test))
    print(f"Treinada nos últimos {fracao:.0%} do treino ({corte} amostras): accuracy teste = {acc_tmp:.2%}")

In [ ]:
# check 4: e se eu forçar uma árvore mais conservadora (mais rasa, folhas
# maiores)? vamos ver se isso resolve o problema.
grade_arvore = {
    "max_depth": [2, 3, 4, 5],
    "min_samples_leaf": [10, 30, 50, 100],
}

busca_arvore = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    grade_arvore,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1,
)
busca_arvore.fit(X_train, y_train)

print(f"Melhores parâmetros: {busca_arvore.best_params_}")
print(f"Melhor accuracy média (CV): {busca_arvore.best_score_:.2%}")
print(f"Accuracy no holdout: {accuracy_score(y_test, busca_arvore.best_estimator_.predict(X_test)):.2%}")
print(f"Para comparação, baseline em CV: 56.16%")

**Resultado da investigação (números reais deste projeto):**

| Checagem | Resultado |
|---|---|
| Gap treino-teste | 67.19% (treino) vs. 52.13% (teste), gap de 15.06 pontos |
| Tamanho da árvore | 28 folhas, profundidade 5, ~64 amostras/folha em média |
| Instabilidade (janelas de treino) | 52.81% / 46.97% / 52.58%, oscila ~6 pontos só mudando a janela |
| Árvore mais conservadora (tuning) | CV = 56.62% (supera o baseline!) mas holdout = 48.54% (desmorona) |

**Interpretação:**

O gap de 15 pontos confirma overfitting, mas moderado: as folhas não são pequenas
demais (64 amostras em média), então o problema não é memorização ponto a ponto. O
teste de instabilidade é mais revelador: variar apenas qual janela de dados a árvore
vê no treino muda a accuracy no MESMO teste em quase 6 pontos. Isso é a assinatura
clássica de alta variância: o modelo é sensível demais a qual pedaço específico do
passado ele observa.

O achado mais importante é o item 4: a árvore mais rasa e conservadora encontrada
pela busca de hiperparâmetros (`max_depth=2, min_samples_leaf=50`) tem, na MÉDIA dos
5 folds do CV, um desempenho de 56.62%, que supera o baseline (56.16%). Mas essa
mesma árvore, testada no holdout final, cai para 48.54%. Essa discrepância não é uma
contradição: é a prova de que mesmo a árvore "ótima" segundo a média do CV não
generaliza de forma confiável para uma fatia específica e não vista dos dados.

Isso explica por que Random Forest e XGBoost, mesmo sem bater o baseline, tiveram
desempenho mais próximo dele e mais estável entre folds do que a árvore sozinha:
ensembles existem justamente para resolver este problema, combinando muitas árvores
fracas e diluindo a variância que fragiliza uma árvore individual. Neste projeto, a
Decision Tree isolada funciona como um exemplo didático direto de por que ensembles
são preferidos em dados financeiros, onde o sinal é fraco e o ruído é alto.

## 8.2 O período de teste é estatisticamente diferente do período de treino?

Um padrão chamou atenção nas seções anteriores: modelos tunados via CV (Decision
Tree e Random Forest) tiveram desempenho MELHOR que o baseline na média dos folds,
mas PIOR que o baseline no holdout final. Isso se repetiu em dois modelos
diferentes, o que sugere que o problema pode não ser do modelo, e sim do próprio
período de teste ter uma dinâmica diferente do restante da série histórica
(quebra de regime / não-estacionariedade).

As checagens abaixo comparam treino e teste diretamente.

In [ ]:
print(f"Período de treino: {X_train.index.min().date()} a {X_train.index.max().date()}")
print(f"Período de teste:  {X_test.index.min().date()} a {X_test.index.max().date()}")

In [ ]:
retorno_treino = df["return"].loc[X_train.index]
retorno_teste = df["return"].loc[X_test.index]

print("Retorno diário médio:")
print(f"  Treino: {retorno_treino.mean():.4%}")
print(f"  Teste:  {retorno_teste.mean():.4%}")

print("\nVolatilidade diária (desvio padrão do retorno):")
print(f"  Treino: {retorno_treino.std():.4%}")
print(f"  Teste:  {retorno_teste.std():.4%}")

print("\nDistribuição do target (% Sobe):")
print(f"  Treino: {y_train.mean():.2%}")
print(f"  Teste:  {y_test.mean():.2%}")

In [ ]:
# olhando a volatilidade dos últimos 20 dias na série toda, com uma linha
# marcando onde começa o período de teste
vol_movel = df["return"].rolling(20).std()

plt.figure(figsize=(13, 4.5))
plt.plot(vol_movel, color="tab:blue", linewidth=0.9)
plt.axvline(X_test.index.min(), color="red", linestyle="--", label="Início do período de teste")
plt.title("Volatilidade móvel de 20 dias ao longo da série")
plt.ylabel("Desvio padrão do retorno diário (20d)")
plt.legend()
plt.tight_layout()
plt.show()

**Como interpretar:** se a volatilidade média do teste for bastante diferente da
volatilidade do treino, ou se o gráfico mostrar o período de teste caindo numa
região visivelmente mais (ou menos) volátil que a média histórica, isso sustenta a
hipótese de quebra de regime: os padrões que os modelos aprenderam no treino
(inclusive o baseline, que só aprende a proporção histórica de Sobe/Cai) descrevem
um regime de mercado que pode não valer integralmente para o período mais recente.

Isso também tem uma implicação prática importante para qualquer projeto que use
holdout temporal simples: um único corte no fim da série pode, por acaso, cair
num período atípico. Uma validação mais robusta usaria múltiplas janelas de teste
(walk-forward) em vez de um único holdout final, para não tirar conclusões fortes
de uma única fatia do tempo.

## 9. Importância das features

Confirma (ou refuta) a hipótese de que as variáveis macro adicionadas, petróleo e
câmbio, realmente contribuem para a decisão do modelo.

In [ ]:
importancias = pd.Series(
    modelo_otimizado.feature_importances_, index=FEATURES
).sort_values(ascending=False)

plt.figure(figsize=(9, 6))
importancias.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Importância das features - Random Forest otimizado")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()

importancias

## 9.1 Diagnóstico visual do modelo otimizado

A accuracy sozinha esconde *qual tipo* de erro o modelo comete. A matriz de
confusão mostra isso diretamente, e as curvas ROC / Precision-Recall mostram como
o trade-off entre acertos e erros muda conforme o limiar de decisão (por padrão o
modelo usa 0.5, mas isso pode ser ajustado).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ConfusionMatrixDisplay.from_estimator(
    modelo_otimizado, X_test, y_test,
    display_labels=["Cai (0)", "Sobe (1)"], ax=axes[0], colorbar=False,
)
axes[0].set_title("Matriz de confusão")

RocCurveDisplay.from_estimator(modelo_otimizado, X_test, y_test, ax=axes[1])
axes[1].set_title("Curva ROC")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")

PrecisionRecallDisplay.from_estimator(modelo_otimizado, X_test, y_test, ax=axes[2])
axes[2].set_title("Curva Precision-Recall")

plt.tight_layout()
plt.show()

## 9.2 Onde o modelo erra: preço com acertos e erros marcados

Sobrepor os acertos e erros do modelo na própria série de preços ajuda a ver se os
erros se concentram em períodos específicos (ex: alta volatilidade, movimentos
bruscos). Um padrão ali seria útil para decidir se vale restringir a estratégia a
certos regimes de mercado.

In [ ]:
predicoes_test = modelo_otimizado.predict(X_test)
acertos = predicoes_test == y_test.values

plt.figure(figsize=(13, 5))
plt.plot(df["Close"].loc[X_test.index], color="lightgray", label="Preço de fechamento", zorder=1)
plt.scatter(
    X_test.index[acertos], df["Close"].loc[X_test.index][acertos],
    color="tab:green", s=15, label="Previsão correta", zorder=2,
)
plt.scatter(
    X_test.index[~acertos], df["Close"].loc[X_test.index][~acertos],
    color="tab:red", s=15, label="Previsão errada", zorder=2,
)
plt.title("Acertos e erros do modelo sobre a série de preços (período de teste)")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Backtest: a estratégia teria gerado retorno?

Aqui simulamos: comprar PETR4 sempre que o
modelo prever Sobe, e ficar fora do mercado quando prever Cai, comparando o
retorno acumulado dessa estratégia com o de simplesmente comprar e manter
(buy & hold) no mesmo período de teste.

Esta simulação ignora custos de transação e slippage; serve para uma comparação
direcional, não como validação de uma estratégia pronta para operar.

In [ ]:
def backtest_estrategia(modelo, X_test: pd.DataFrame, close: pd.Series,
                         horizonte: int) -> pd.DataFrame:
    """Simula retorno acumulado de uma estratégia 'comprar quando o modelo prever Sobe'."""
    retorno_futuro = close.shift(-horizonte) / close - 1
    retorno_futuro_test = retorno_futuro.loc[X_test.index]

    predicoes = modelo.predict(X_test)
    retorno_estrategia = np.where(predicoes == 1, retorno_futuro_test, 0.0)

    curvas = pd.DataFrame({
        "estrategia": (1 + pd.Series(retorno_estrategia, index=X_test.index)).cumprod(),
        "buy_and_hold": (1 + retorno_futuro_test).cumprod(),
    })
    return curvas

In [ ]:
curvas_backtest = backtest_estrategia(modelo_otimizado, X_test, df["Close"], HORIZON_DIAS)

plt.figure(figsize=(10, 5))
plt.plot(curvas_backtest["estrategia"], label="Estratégia (segue o modelo)")
plt.plot(curvas_backtest["buy_and_hold"], label="Buy & Hold")
plt.title("Retorno acumulado no período de teste")
plt.ylabel("Capital (base 1.0)")
plt.legend()
plt.tight_layout()
plt.show()

retorno_estrategia_total = curvas_backtest["estrategia"].iloc[-1] - 1
retorno_buy_hold_total = curvas_backtest["buy_and_hold"].iloc[-1] - 1
print(f"Retorno acumulado - estratégia: {retorno_estrategia_total:.2%}")
print(f"Retorno acumulado - buy & hold: {retorno_buy_hold_total:.2%}")

## 11. Conclusões

### O que o projeto se propôs a responder

É possível prever se a PETR4 vai fechar em alta ou em baixa daqui a 5 dias úteis,
usando indicadores técnicos e variáveis macro (Ibovespa, petróleo Brent, câmbio
USD/BRL)?

### Resultado principal: nenhum modelo supera de forma confiável um baseline ingênuo

| Modelo | CV médio (5 folds, TimeSeriesSplit) |
|---|---|
| Baseline (sempre prever a classe majoritária) | 56,16% (+/- 2,17%) |
| Random Forest | 55,46% (+/- 2,51%) |
| XGBoost | 53,95% (+/- 3,15%) |
| Decision Tree | 49,14% (+/- 3,87%) |

Mesmo depois da otimização de hiperparâmetros via GridSearchCV, os modelos ajustados
melhoraram a média do CV (Random Forest ajustado: 58,18%; Decision Tree ajustado:
56,62%), mas pioraram no holdout final (49,21% e 48,54%, respectivamente), os dois
abaixo do baseline no mesmo período (50,79%).

### Por que isso acontece: quebra de regime, não falha do modelo

A investigação da seção 8.2 explica a causa raiz. O período de treino (2017-2024) e
o período de teste (2024-2026) têm dinâmicas de mercado diferentes:

| | Treino | Teste |
|---|---|---|
| Retorno diário médio | 0,16% | 0,06% |
| Volatilidade diária | 2,77% | 1,53% |
| % de períodos "Sobe" (5d) | 58,60% | 50,79% |

Durante 2017-2024, a PETR4 teve uma tendência de alta persistente (quase 59% dos
períodos de 5 dias fechavam em alta). Isso não é previsibilidade técnica real, é
apenas a ação estando em alta geral naquele período. O baseline e os modelos
absorveram essa tendência de fundo durante o treino. Já o período mais recente
(2024-2026) foi visivelmente mais calmo e mais equilibrado (quase 50/50 entre Sobe
e Cai, com metade da volatilidade). Quando essa tendência de fundo desaparece, não
sobra nenhum modelo com vantagem preditiva real, nem os "sofisticados" nem o
baseline ingênuo.

Esse mesmo fenômeno explica a discrepância entre CV e holdout dos modelos
ajustados: a busca de hiperparâmetros otimiza para a média histórica de vários
períodos (que inclui a fase de tendência forte), mas essa média não descreve bem o
período de teste, que é estruturalmente diferente.

### O que isso ensina sobre o problema (e sobre validação de modelos)

Accuracy sem baseline não significa nada. Um modelo com 55% de accuracy parece
razoável isoladamente, mas perde para "não fazer nada" quando comparado
corretamente.

CV médio pode esconder problemas de um período específico. Um único holdout no
fim da série pode, por acaso (ou não por acaso, como aqui), cair num regime
atípico. Uma validação mais robusta usaria múltiplas janelas de teste
(walk-forward) para não tirar conclusões fortes de uma única fatia do tempo.

Séries financeiras não são estacionárias. Um padrão que existiu num período
(tendência de alta persistente) não necessariamente se repete, isso é consistente
com a hipótese de mercado eficiente e com a dificuldade conhecida de prever
retornos de curto prazo com indicadores técnicos.

Uma árvore de decisão única é mais instável do que ensembles (seção 8.1): a mesma
árvore treinada em janelas de treino ligeiramente diferentes variou quase 6 pontos
percentuais de accuracy no mesmo teste. Random Forest e XGBoost, ao combinar muitas
árvores fracas, ficaram mais próximos do baseline e mais estáveis entre folds, mas
não imunes ao problema de fundo (a quebra de regime).

### Limitações

O backtest de estratégia (seção 10) e a análise de importância de features (seção
9) ainda precisam ser executados com os dados finais para completar o quadro. Em
particular, vale checar se as variáveis macro (Brent, USD/BRL) tiveram peso
relevante no modelo, o que sustentaria a tese de que contexto externo ajuda mesmo
quando o modelo como um todo não supera o baseline.

A comparação de holdout usa uma única janela de teste; os números de CV são mais
confiáveis como estimativa de desempenho médio, mas mesmo eles cobrem
majoritariamente o regime de alta de 2017-2024.

O backtest, quando executado, deve ser lido com a ressalva de que ignora custos de
transação e slippage.

